# Merge pairs — pRESTO AssemblePairs

Generic paired-read assembly notebook for rima datasets. It merges already trimmed FASTQ reads with `AssemblePairs.py align --coord illumina --rc tail --failed`.

Recommended input for sheep `PRJNA900592` is `label='pr_trimmed'`, i.e. `results/PRJNA900592/pr_trimmed/fastq`. This notebook does **not** run FastQC/MultiQC; merged QC is a separate `qc.ipynb` step with `label='merged'`.


In [ ]:
import os, sys, sysconfig
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)



In [ ]:
import gzip, shutil, subprocess
from pathlib import Path


def fastq_dir(volume, dataset, label):
    vol = Path(volume)
    if label == "trimmed":
        return vol / "results" / dataset / "trimmed" / "fastq"
    if label == "pr_trimmed":
        return vol / "results" / dataset / "pr_trimmed" / "fastq"
    raise ValueError("label must be 'trimmed' or 'pr_trimmed' for merge_pairs")


def discover_pairs(input_dir):
    input_dir = Path(input_dir)
    patterns = [
        ("*_1.pr.fastq.gz", "_1.pr.fastq.gz", "_2.pr.fastq.gz"),
        ("*_1.trim.fastq.gz", "_1.trim.fastq.gz", "_2.trim.fastq.gz"),
    ]
    pairs = []
    seen = set()
    for glob_pat, r1_suffix, r2_suffix in patterns:
        for r1 in sorted(input_dir.glob(glob_pat)):
            sample = r1.name[:-len(r1_suffix)]
            if sample in seen:
                continue
            r2 = input_dir / f"{sample}{r2_suffix}"
            if not r2.exists():
                raise FileNotFoundError(f"Missing mate for {r1}: expected {r2}")
            pairs.append((sample, r1, r2))
            seen.add(sample)
    return pairs


def count_fastq_records(path):
    path = Path(path)
    if not path.exists():
        return 0
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt") as handle:
        return sum(1 for _ in handle) // 4



In [ ]:
def run_merge_pairs(volume, dataset, label="pr_trimmed", force=False):
    # Merge paired FASTQ reads with pRESTO AssemblePairs.
    # Input labels:
    #   pr_trimmed -> results/<dataset>/pr_trimmed/fastq/*.pr.fastq.gz
    #   trimmed    -> results/<dataset>/trimmed/fastq/*.trim.fastq.gz
    # Output:
    #   results/<dataset>/merged/fastq, logs, qc
    if not shutil.which("AssemblePairs.py"):
        raise RuntimeError("AssemblePairs.py not found. Use BCR Pipeline kernel or run setup_vm_conda.sh.")

    vol = Path(volume)
    input_dir = fastq_dir(vol, dataset, label)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input FASTQ directory not found: {input_dir}")

    out_base = vol / "results" / dataset / "merged"
    out_fastq = out_base / "fastq"
    out_logs = out_base / "logs"
    out_qc = out_base / "qc"
    for d in (out_fastq, out_logs, out_qc):
        d.mkdir(parents=True, exist_ok=True)

    pairs = discover_pairs(input_dir)
    if not pairs:
        raise FileNotFoundError(f"No paired FASTQ files found in {input_dir}")

    print(f"[merge_pairs] dataset={dataset} label={label}")
    print(f"[merge_pairs] input={input_dir}")
    print(f"[merge_pairs] pairs={len(pairs)}")

    results = []
    for sample, r1, r2 in pairs:
        pass_fastq = out_fastq / f"{sample}_assemble-pass.fastq"
        fail_fastq = out_fastq / f"{sample}_assemble-fail.fastq"
        log_path = out_logs / f"{sample}.assemble.log"
        stdout_path = out_logs / f"{sample}.stdout.txt"
        stderr_path = out_logs / f"{sample}.stderr.txt"

        if pass_fastq.exists() and not force:
            print(f"  [skip] {sample}: {pass_fastq.name} exists")
            status = "skipped"
        else:
            cmd = [
                "AssemblePairs.py", "align",
                "-1", str(r1),
                "-2", str(r2),
                "--coord", "illumina",
                "--rc", "tail",
                "--outname", sample,
                "--outdir", str(out_fastq),
                "--log", str(log_path),
                "--failed",
            ]
            print(f"  [run] {sample}")
            res = subprocess.run(cmd, capture_output=True, text=True)
            stdout_path.write_text(res.stdout)
            stderr_path.write_text(res.stderr)
            if res.returncode != 0:
                raise RuntimeError(f"AssemblePairs failed for {sample}; see {stderr_path}")
            status = "done"

        results.append((sample, status, pass_fastq, fail_fastq, log_path))

    manifest_path = out_qc / "assemble_manifest.tsv"
    with open(manifest_path, "w") as out:
        out.write("sample\tstatus\tpass_fastq\tfail_fastq\tlog\n")
        for sample, status, pass_fastq, fail_fastq, log_path in results:
            out.write(f"{sample}\t{status}\t{pass_fastq}\t{fail_fastq}\t{log_path}\n")

    qc_path = out_qc / "assembly_qc.tsv"
    total_pass = 0
    total_fail = 0
    with open(qc_path, "w") as out:
        out.write("sample\tassembled_pairs\tfailed_pairs\ttotal_pairs_seen\tmerge_rate\tpass_fastq\tfail_fastq\tlog\n")
        for sample, status, pass_fastq, fail_fastq, log_path in results:
            n_pass = count_fastq_records(pass_fastq)
            n_fail = count_fastq_records(fail_fastq)
            total = n_pass + n_fail
            rate = n_pass / total if total else 0
            total_pass += n_pass
            total_fail += n_fail
            out.write(f"{sample}\t{n_pass}\t{n_fail}\t{total}\t{rate:.6f}\t{pass_fastq}\t{fail_fastq}\t{log_path}\n")

    denom = total_pass + total_fail
    print(f"[merge_pairs] manifest: {manifest_path}")
    print(f"[merge_pairs] qc:       {qc_path}")
    print(f"[merge_pairs] assembled_pairs={total_pass:,}")
    print(f"[merge_pairs] failed_pairs={total_fail:,}")
    print(f"[merge_pairs] merge_rate={(total_pass / denom if denom else 0):.4f}")
    print("[merge_pairs] For merged QC, run qc.ipynb with label='merged'.")
    return qc_path



### Run

Run sheep first. For other datasets, change only `dataset` and `label`.

- sheep after technical-primer trim: `label='pr_trimmed'`
- datasets without primer trim: `label='trimmed'`


In [ ]:
run_merge_pairs('/data/user/epishkin', 'PRJNA900592', label='pr_trimmed', force=False)
